In [2]:
import os  # Lets us choose Java before Spark starts.
import shutil  # Finds Ubuntu's java command.
from pathlib import Path  # Represents filesystem paths.
from pyspark.sql import SparkSession  # Creates a Spark session.
from delta import configure_spark_with_delta_pip  # Adds Delta's Java libraries.

java_command = shutil.which("java")  # Find java in Ubuntu.
if java_command is None:  # Check that Java was found.
    raise RuntimeError("Java was not found")  # Stop with a clear error if it was not.

java_home = Path(java_command).resolve().parent.parent  # Find Java's installation folder.
os.environ["JAVA_HOME"] = str(java_home)  # Point Spark to Ubuntu's Java.

project_root = Path("/mnt/f/Big Data/Retail_Lakehouse_Learning")  # Locate our existing project.

builder = (  # Set up Spark before starting it.
    SparkSession.builder
    .master("local[2]")  # Use two local processing threads.
    .appName("RetailBronzeCSV")  # Name this notebook's Spark session.
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")  # Enable Delta SQL.
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")  # Enable Delta tables.
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()  # Start Spark with Delta.

print("Project found:", project_root.is_dir())  # Expect True.
print("Spark version:", spark.version)  # Expect 4.1.1.

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/09 16:40:36 WARN Utils: Your hostname, USMAN-ALI, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/09 16:40:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/usmanali611b/.venvs/retail-lakehouse/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/usmanali611b/.ivy2.5.2/cache
The jars for the packages stored in: /home/usmanali611b/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b32f6db3-270b-4aa5-8299-f83e7c1c6bf2;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.3.1 in central
	found io.delta#delta-storage;4.3.1 in central
	found io.unitycatalog#unitycatalog-client;0.5.0 in central
	found org.slf4j#slf4j-api;2.0.1

Project found: True
Spark version: 4.1.1


In [3]:
source_files = {  # Give each source a table name and its existing CSV path.
    "products": project_root / "landing" / "flat_files" / "products.csv",  # Product details.
    "stores": project_root / "landing" / "flat_files" / "stores.csv",  # Store details.
    "promotions": project_root / "landing" / "flat_files" / "promotions.csv",  # Promotion details.
    "suppliers": project_root / "landing" / "flat_files" / "suppliers.csv",  # Supplier details.
    "weather_fallback": project_root / "landing" / "weather_fallback" / "weather_sample.csv",  # Weather sample.
}

for table_name, csv_path in source_files.items():  # Check each table name and its file path.
    print(table_name, "exists:", csv_path.is_file())  # Show whether that source file exists.

products exists: True
stores exists: True
promotions exists: True
suppliers exists: True
weather_fallback exists: True


In [4]:
expected_rows = {  # Record the row counts we verified from the source workbook.
    "products": 120,  # Expected product rows.
    "stores": 6,  # Expected store rows.
    "promotions": 15,  # Expected promotion rows.
    "suppliers": 18,  # Expected supplier rows.
    "weather_fallback": 552,  # Expected weather rows.
}

for table_name, csv_path in source_files.items():  # Visit each existing source CSV.
    source_df = (  # Create a Spark DataFrame for this file.
        spark.read  # Start a Spark read operation.
        .option("header", "true")  # Use the first CSV row as column names.
        .option("inferSchema", "false")  # Keep source values as strings in Bronze.
        .csv(str(csv_path))  # Read the CSV at this path.
    )
    actual_rows = source_df.count()  # Count its data rows.
    expected_count = expected_rows[table_name]  # Look up the count we expect.
    print(f"{table_name}: {actual_rows}/{expected_count} rows")  # Display both counts.

    if actual_rows != expected_count:  # Stop if a source has changed unexpectedly.
        raise ValueError(f"Unexpected row count for {table_name}")  # Show which source needs checking.

products: 120/120 rows
stores: 6/6 rows
promotions: 15/15 rows
suppliers: 18/18 rows
weather_fallback: 552/552 rows


In [5]:
for table_name, csv_path in source_files.items():  # Process each of the five source CSVs.
    source_df = (  # Read this source into a Spark DataFrame.
        spark.read  # Start a Spark read operation.
        .option("header", "true")  # Take column names from the first row.
        .option("inferSchema", "false")  # Preserve source values as strings.
        .csv(str(csv_path))  # Read the existing CSV file.
    )

    source_count = source_df.count()  # Count rows before writing.
    if source_count != expected_rows[table_name]:  # Check against our verified count.
        raise ValueError(f"Unexpected source row count: {table_name}")  # Stop before writing changed data.

    output_path = project_root / "lakehouse" / "bronze" / "data" / table_name  # Choose this Delta table's folder.
    source_df.write.format("delta").mode("overwrite").save(str(output_path))  # Save a complete Bronze snapshot.

    saved_df = spark.read.format("delta").load(str(output_path))  # Read the saved Delta table back.
    saved_count = saved_df.count()  # Count rows after writing.
    if saved_count != source_count:  # Check that no rows were lost.
        raise ValueError(f"Bronze row count mismatch: {table_name}")  # Stop and identify the table.

    print(f"{table_name}: source={source_count}, bronze={saved_count}")  # Show the verified result.

26/10/09 16:41:05 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

products: source=120, bronze=120


stores: source=6, bronze=6


promotions: source=15, bronze=15


suppliers: source=18, bronze=18


weather_fallback: source=552, bronze=552


In [6]:
products_path = project_root / "lakehouse" / "bronze" / "data" / "products"  # Locate the products Delta table.
products_df = spark.read.format("delta").load(str(products_path))  # Read the committed table.
print("Saved product rows:", products_df.count())  # Check how many rows it contains.

Saved product rows: 120
